In [53]:
import pandas as pd

##### Tratamento do nome dos países para ficarem iguais em todos os datasets

*Olimpiadas*  
Trata o texto do nome das olimpiadas pois estava diferente do que tem no pib, estava: USA Estados Unidos, após o tratamento fica uma coluna para o país que tem só o nome: Estados Unidos e outra coluna para a NOC que recebe a sigla (talvez use no futuro para comparar com a tabela de atletas em outras analises).

In [54]:
df_olimpiadas = pd.read_csv('medalhas_olimpiadas_wikipedia.csv', encoding='utf-8')
display(df_olimpiadas.head())

df_paraolimpiadas = pd.read_csv('medalhas_paralimpiadas_wikipedia.csv', encoding='utf-8')
display(df_paraolimpiadas.head())

,No.,País,Jogos,Ouro,Prata,Bronze,Total
0,1,USA Estados Unidos,29,1101,880,781,2762
1,2,URS União Soviética,9,395,319,296,1010
2,3,CHN China,12,302,227,197,726
3,4,GBR Grã-Bretanha,30,300,338,344,982
4,5,FRA França,30,240,280,298,818


,País,№,Ouro,Prata,Bronze,Total
0,África do Sul (RSA),11,121,95,88,304
1,Alemanha (GER) [a],8,199,266,253,718
2,Alemanha Ocidental (FRG) [b],8,322,260,246,828
3,Alemanha Oriental (GDR) [c],1,0,3,1,4
4,Angola (ANG),6,4,3,1,8


olimpiadas

In [55]:
df_olimpiadas_limpo = df_olimpiadas['País'].str.extract(r'^([A-Z]{3})\s+(.*)')
df_olimpiadas_limpo.columns = ['NOC', 'País_Limpo']

df_olimpiadas['NOC'] = df_olimpiadas_limpo['NOC']
df_olimpiadas['País'] = df_olimpiadas_limpo['País_Limpo']

df_olimpiadas['País'] = df_olimpiadas['País'].str.replace(r'\[.*\]|\*', '', regex=True).str.strip()

display(df_olimpiadas.head())

,No.,País,Jogos,Ouro,Prata,Bronze,Total,NOC
0,1,Estados Unidos,29,1101,880,781,2762,USA
1,2,União Soviética,9,395,319,296,1010,URS
2,3,China,12,302,227,197,726,CHN
3,4,Grã-Bretanha,30,300,338,344,982,GBR
4,5,França,30,240,280,298,818,FRA


In [56]:
df_olimpiadas.to_csv('olimpiadas_medalhas_arrumado.csv', index=False, encoding='utf-8')

*Trata para as paralimpiadas*

trata o nome do país para ficar igual aos do pib, estava com o nome do país e o código entre parênteses, agora só tem o nome do paíse o codigo do país em uma coluna separada, isso facilita a comparação entre os dataframes

paralimpiadas

In [57]:
df_paraolimpiadas['NOC'] = df_paraolimpiadas['País'].str.extract(r'\((.*?)\)')

df_paraolimpiadas['País'] = df_paraolimpiadas['País'].str.replace(r'\s*\(.*', '', regex=True)
df_paraolimpiadas['País'] = df_paraolimpiadas['País'].str.replace(r'\[.*\]', '', regex=True).str.strip()
display(df_paraolimpiadas.head())

,País,№,Ouro,Prata,Bronze,Total,NOC
0,África do Sul,11,121,95,88,304,RSA
1,Alemanha,8,199,266,253,718,GER
2,Alemanha Ocidental,8,322,260,246,828,FRG
3,Alemanha Oriental,1,0,3,1,4,GDR
4,Angola,6,4,3,1,8,ANG


Junta as Tres alemanhas paralimpicas em uma só, soma as medalhas e usa o código GER.

In [59]:
df_paraolimpiadas.info()
display(list(df_paraolimpiadas.columns))  

<class 'pandas.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   País    132 non-null    str  
 1   №       132 non-null    int64
 2   Ouro    132 non-null    int64
 3   Prata   132 non-null    int64
 4   Bronze  132 non-null    int64
 5   Total   132 non-null    int64
 6   NOC     131 non-null    str  
dtypes: int64(5), str(2)
memory usage: 7.3 KB


['País', '№', 'Ouro', 'Prata', 'Bronze', 'Total', 'NOC']

In [60]:
alemanhas = ['Alemanha Ocidental', 'Alemanha Oriental']
df_paraolimpiadas['País'] = df_paraolimpiadas['País'].replace(alemanhas, 'Alemanha')

# 2. Agrupar mantendo TODAS as colunas de medalhas
# Usamos 'sum' para os números e 'first' para a sigla (NOC)
df_paraolimpiadas = df_paraolimpiadas.groupby('País').agg({
    'Ouro': 'sum',
    'Prata': 'sum',
    'Bronze': 'sum',
    'Total': 'sum',
    'NOC': 'first' 
}).reset_index()

# 3. Ordenar pelo total para conferir o ranking
df_paraolimpiadas = df_paraolimpiadas.sort_values(by='Total', ascending=False)

display(df_paraolimpiadas.head(10))

,País,Ouro,Prata,Bronze,Total,NOC
115,Totais,7513,7298,7336,22147,NaN
40,Estados Unidos,808,736,739,2283,USA
48,Grã-Bretanha,667,621,626,1914,GBR
0,Alemanha,521,529,500,1550,GER
22,China,535,400,302,1237,CHN
6,Austrália,389,422,394,1205,AUS
46,França,357,362,373,1092,FRA
17,Canadá,400,340,346,1080,CAN
93,Países Baixos,301,262,245,808,NED
95,Polônia,269,265,220,754,POL


Exclui a linha que tinhas a soma de todas as medalhas

In [61]:
df_paraolimpiadas = df_paraolimpiadas[df_paraolimpiadas['País'] != 'Totais']

display(df_paraolimpiadas)

,País,Ouro,Prata,Bronze,Total,NOC
40,Estados Unidos,808,736,739,2283,USA
48,Grã-Bretanha,667,621,626,1914,GBR
0,Alemanha,521,529,500,1550,GER
22,China,535,400,302,1237,CHN
6,Austrália,389,422,394,1205,AUS
...,...,...,...,...,...,...
87,Omã,0,0,1,1,OMA
107,Sudão,1,0,0,1,SUD
112,Síria,0,0,1,1,SYR
5,Atletas Paralímpicos Neutros,0,0,0,0,NPA


In [62]:
df_paraolimpiadas.to_csv('paralimpiada_medalhas_arrumado.csv', index=False, encoding='utf-8')